In [ ]:
import collections
import os
import re

import numpy as np
import yaml
# import PIL
# import PIL.Image
import tensorflow as tf

import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import layers, Sequential
import tensorflow.keras.backend as K



In [ ]:
# 精度がばらけるので再現性確認のため、cpuだけで学習する
# tf.config.set_visible_devices([], "GPU")

In [ ]:
# yamlの設定ファイルの読み込み

with open("../config/config.yaml", "rb") as f:
    config = yaml.safe_load(f)
    # print(config)
    # print("launch json modified")
    # print("mapping_namelist", config["class_mapping"]["name_list"])


In [ ]:
# dir_data = config["path_extracted_data"]

dir_path_train = config["path_split_extracted_data"]["train"]
dir_path_val = config["path_split_extracted_data"]["val"]
dir_path_test = config["path_split_extracted_data"]["common_test"]

# 画像サイズは、efficientnet b0の想定
img_height = 224
img_width = 224

seed = 42
batch_size = 16

split_rate = 0.2

epoch = 15


1.画像データの読み込み

In [ ]:
# 学習データの読み込み
dataset_train = tf.keras.utils.image_dataset_from_directory(
    dir_path_train,
    # validation_split = split_rate,
    # subset = "training",    # 学習用    
    seed = seed, 
    image_size = (img_height, img_width),
    batch_size=batch_size,
    shuffle=False,
)

In [ ]:
# 検証データの読み込み
dataset_val = tf.keras.utils.image_dataset_from_directory(
    dir_path_val,
    # validation_split = split_rate,
    # subset = "val",    # 検証用    
    seed = seed, 
    image_size = (img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# 共通テストデータの読み込み
dataset_test = tf.keras.utils.image_dataset_from_directory(
    dir_path_test,
    # validation_split = split_rate,
    # subset = "test",    # テスト用    
    seed = seed, 
    image_size = (img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

2.class weightを追加したモデル

2.1 class weightの計算

In [ ]:
# modelのfitの時に、class_wightを渡すための準備

# 学習データのクラスごとの偏りの確認。
# dir_path_train
list_class_names = config["class_mapping"]["name_list"]
dict_class_counts = {}

for class_name in list_class_names:
    fp = os.path.join(dir_path_train, class_name)
    
    list_file_names = [
            f for f in os.listdir(fp)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ]
    
    dict_class_counts[class_name] = len(list_file_names)
    
    print(f"{class_name}:", len(list_file_names))



In [ ]:
dict_class_counts

In [ ]:
dataset_train.class_names

In [ ]:
# 学習データの、ディレクトリ名のクラス名称と、データセット内部でone hot encodingされた数字の対応表
list_dataset_class_names = dataset_train.class_names

dict_class_map = {i: name for i, name in enumerate(list_dataset_class_names)}
print("対応表:", dict_class_map)

In [ ]:
# clss_weightの作成

y = []

for k in dict_class_map:
    # print(dict_class_counts[dict_class_map[k]])

    y.extend([int(k)]*dict_class_counts[dict_class_map[k]])

classes = np.unique(y)

weights = compute_class_weight(
    class_weight = "balanced", 
    classes = classes,
    y=y
)

dict_class_wights = {cl: weight for cl, weight in zip(classes, weights)}

In [ ]:
weights

In [ ]:
dict_class_wights

2.2class weightを追加したモデルの評価

In [ ]:
# 再現性確保のため、
# 乱数設定〜学習〜メモリ解放までを1セルで管理

#乱数設定
tf.keras.utils.set_random_seed(seed)

# TensorFlowの「決定論的動作」を設定。最終版のモデルの学習時はコメントアウト
tf.config.experimental.enable_op_determinism()

# top層を除いた、転移学習元のモデルの読み込み
basemodel = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(img_height, img_width, 3))

# トップ層のみ学習したいため、層を凍結
basemodel.trainable = False

# モデルの構築
inputs = layers.Input(shape=(img_height, img_width, 3))
x = basemodel(inputs, training=False) #base_model.trainable=Falseしてるけど、batch nomarization層を固定するためにここでもfalse
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(len(dataset_train.class_names), activation="softmax")(x) #多分類なのでsoftmax

model_weight = Model(inputs, outputs)

model_weight.compile(
    optimizer="adam",
    # optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("train starts")
model_weight.fit(
    x=dataset_train, 
    epochs=epoch, 
    validation_data=dataset_val,
    class_weight=dict_class_wights
    )

# tf.keras.utils.image_dataset_from_directoryでデータセットを作ると、ディレクトリ名がそのまま正解ラベルになる。
# けど、predictでは、正解ラベルを無視して予測してくれる

print("\npred starts")
y_probs_weight = model_weight.predict(dataset_test)

# 各カテゴリの確率のうち、最も高い確率のカテゴリを予測されたカテゴリとする
y_pred_weight = np.argmax(y_probs_weight, axis=-1)

# バッチごとの、xが画像の要素、yがラベル。yを正解ラベルとして取り出す。
y_true = np.concatenate([y for x, y in dataset_test], axis=0)

acc_weight = accuracy_score(y_true, y_pred_weight)
print("acc_weight:",acc_weight)

K.clear_session()


In [ ]:
# 混合行列の確認
confusion_matrix(y_true, y_pred_weight)

In [ ]:
# 全クラスの平均をとる
f1_score(y_true, y_pred_weight, average="macro")

In [ ]:
classification_report(y_true, y_pred_weight, target_names=dataset_test.class_names)

3 optimizerを変更したモデル

3.1optimizerを変更したモデルの性能調査

In [ ]:
# 再現性確保のため、
# 乱数設定〜学習〜メモリ解放までを1セルで管理

#乱数設定
tf.keras.utils.set_random_seed(seed)

# TensorFlowの「決定論的動作」を設定。最終版のモデルの学習時はコメントアウト
tf.config.experimental.enable_op_determinism()

# top層を除いた、学習済みモデルの読み込み

model_sgd = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(img_height, img_width, 3))

# トップ層のみ学習したいため、層を凍結
model_sgd.trainable = False

# モデルの構築
inputs = layers.Input(shape=(img_height, img_width, 3))
x = model_sgd(inputs, training=False) #base_model.trainable=Falseしてるけど、batch nomarization層を固定するためにここでもfalse
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(len(dataset_train.class_names), activation="softmax")(x) #多分類なのでsoftmax

model_sgd = Model(inputs, outputs)

model_sgd.compile(
    # optimizer="sgd",
    optimizer=SGD(),
    # optimizer=SGD(momentum=0.9),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_sgd.fit(
    x=dataset_train, 
    epochs=epoch, 
    validation_data=dataset_val,
    )

# tf.keras.utils.image_dataset_from_directoryでデータセットを作ると、ディレクトリ名がそのまま正解ラベルになる。
# けれど、predictでは、正解ラベルを無視して予測してくれる

y_probs_sgd = model_sgd.predict(dataset_test)

# 各カテゴリの確率のうち、最も高い確率のカテゴリを予測されたカテゴリとする
y_pred_sgd = np.argmax(y_probs_sgd, axis=-1)

# バッチごとの、xが画像の要素、yがラベル。yを正解ラベルとして取り出す。
y_true = np.concatenate([y for x, y in dataset_test], axis=0)

acc_sgd = accuracy_score(y_true, y_pred_sgd)
print("acc_sgd", acc_sgd)

K.clear_session()

In [ ]:
# 混合行列の確認
confusion_matrix(y_true, y_pred_sgd)

In [ ]:
# 全クラスの平均をとる
f1_score(y_true, y_pred_sgd, average="macro")

4.データ拡張を追加したモデル

4.1データ拡張層を追加したモデルの性能評価

In [ ]:
# 再現性確保のため、
# 乱数設定〜学習〜メモリ解放までを1セルで管理

#乱数設定
tf.keras.utils.set_random_seed(seed)

# TensorFlowの「決定論的動作」を設定。最終版のモデルの学習時はコメントアウト
tf.config.experimental.enable_op_determinism()

# top層を除いた、学習済みモデルの読み込み

model_da = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(img_height, img_width, 3))

# トップ層のみ学習したいため、層を凍結
model_da.trainable = False

# データ拡張レイヤーを定義*推論時には無効になる
data_augmentation = Sequential([
    # 水平または垂直にランダムに反転
    layers.RandomFlip("horizontal_and_vertical"),
    # 最大20%（0.2）の範囲でランダムに回転
    layers.RandomRotation(0.2),
])

# モデルの構築
inputs = layers.Input(shape=(img_height, img_width, 3))
x = model_da(inputs, training=False) #base_model.trainable=Falseしてるけど、batch nomarization層を固定するためにここでもfalse
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(len(dataset_train.class_names), activation="softmax")(x) #多分類なのでsoftmax

model_da = Model(inputs, outputs)

model_da.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_da.fit(
    x=dataset_train, 
    epochs=epoch, 
    validation_data=dataset_val,
    )

# tf.keras.utils.image_dataset_from_directoryでデータセットを作ると、ディレクトリ名がそのまま正解ラベルになる。
# けど、predictでは、正解ラベルを無視して予測してくれる
y_probs_da = model_da.predict(dataset_test)

# 各カテゴリの確率のうち、最も高い確率のカテゴリを予測されたカテゴリとする
y_pred_da = np.argmax(y_probs_da, axis=-1)

# バッチごとの、xが画像の要素、yがラベル。yを正解ラベルとして取り出す。
y_true = np.concatenate([y for x, y in dataset_test], axis=0)

acc_da = accuracy_score(y_true, y_pred_da)
print("acc_da:",acc_da)

K.clear_session()


In [ ]:
# 混合行列の確認
confusion_matrix(y_true, y_pred_da)

In [ ]:
# 全クラスの平均をとる
f1_score(y_true, y_pred_da, average="macro")

readme用の画像作成

In [ ]:
# 精度表

# データ
exp_ids = ["exp01", "exp02", "exp03", "exp04"]
accuracy = [0.9381, 0.9278, 0.9278, 0.9278]
f1 = [0.9424, 0.9300, 0.9352, 0.9300]

# プロット
plt.figure()
plt.bar(exp_ids, accuracy)
plt.title("Accuracy Comparison")
plt.xlabel("Experiment")
plt.ylabel("Accuracy")
plt.show()

plt.figure()
plt.bar(exp_ids, f1)
plt.title("F1 Score Comparison")
plt.xlabel("Experiment")
plt.ylabel("F1 Score")
plt.show()

In [ ]:
# val lossの推移

# データ
baseline = [1.3766,0.5126,0.3063,0.2449,0.2132,0.1872,0.1745,0.1692,0.1581,0.1518,0.1557,0.1564,0.1529,0.1502,0.1394]
sgd = [12.1933,9.8504,3.2854,1.2403,0.6354,0.4588,0.4290,0.3711,0.4157,0.3215,0.3374,0.3351,0.4233,0.2971,0.2735]

epochs = range(1, len(baseline)+1)

plt.figure()
plt.plot(epochs, baseline, label="Adam (baseline)")
plt.plot(epochs, sgd, label="SGD")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Curve")
plt.legend()

plt.show()

In [ ]:
# 実験結果表
# データ
exp_ids = ["exp01", "exp02", "exp03", "exp04"]
accuracy = [0.9381, 0.9278, 0.9278, 0.9278]
f1 = [0.9424, 0.9300, 0.9352, 0.9300]

# プロット
plt.figure()
plt.bar(exp_ids, accuracy)
plt.title("Accuracy Comparison")
plt.xlabel("Experiment")
plt.ylabel("Accuracy")
plt.show()

plt.figure()
plt.bar(exp_ids, f1)
plt.title("F1 Score Comparison")
plt.xlabel("Experiment")
plt.ylabel("F1 Score")
plt.show()